# mambabedding (1)

This notebook documents a computer vision experiment, including data preparation, model setup, and evaluation steps.

This file keeps the original experimentation style and adds a small structure note for easier navigation.


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
from transformers import (
    AutoModelForImageClassification,
    AutoImageProcessor,
    AutoModel,
    PretrainedConfig,
    PreTrainedModel
)
from transformers.image_utils import load_image
from peft import LoraConfig, get_peft_model
from PIL import Image
import random
import os
from urllib.request import urlopen
from PIL import Image
import timm
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
import matplotlib.pyplot as plt

import torch

import cv2

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

2026-03-22 04:45:40.288617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774154740.310827    1664 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774154740.317496    1664 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774154740.334762    1664 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774154740.334783    1664 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774154740.334785    1664 computation_placer.cc:177] computation placer alr

In [3]:
train=pd.read_csv("/kaggle/input/ucf101-action-recognition/train.csv")
evalr=pd.read_csv("/kaggle/input/ucf101-action-recognition/val.csv")

In [4]:
modelr="timm/tiny_vit_21m_224.dist_in22k_ft_in1k"
image_processor = AutoImageProcessor.from_pretrained(modelr)


In [5]:
class BidirectionalMambaLayer(nn.Module):
    def __init__(self, d_model, d_state):
        super().__init__()
        self.D = d_model
        self.N = d_state

        # Input projection → split into SSM input and gate
        self.in_proj = nn.Linear(d_model, d_model)

        # Selective SSM parameter projections
        self.delta_proj = nn.Linear(d_model, d_model)
        self.u_time=torch.nn.Conv1d(in_channels=d_model,out_channels=d_model,kernel_size=3,groups=d_model,padding=1)
        self.B_proj     = nn.Linear(d_model, d_model * d_state)
        self.C_proj     = nn.Linear(d_model, d_model * d_state)

        # Diagonal A (one per channel, per state)
        self.A = nn.Parameter(-torch.rand(d_model, d_state))

        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)
  
        

    def forward(self, x):
        B, L, D = x.shape

        # ---- 1. Input projection ----
        u= self.in_proj(x)
        u=self.u_time(u.transpose(1,2)).transpose(1,2)
                        # (B, L, D)

        # ---- 2. Selective parameters ----
        delta = nn.functional.softplus(self.delta_proj(u))     # (B, L, D)
        B_t   = self.B_proj(u).view(B, L, D, self.N)
        C_t   = self.C_proj(u).view(B, L, D, self.N)

        # ---- 3. SSM scan (sequential for clarity) ----
        s = torch.zeros(B, D, self.N, device=x.device)               # initial state
        outputs = []

        for t in range(L):
            expA = torch.exp(delta[:, t].unsqueeze(-1) * self.A)

            s = expA * s + B_t[:, t]                # state update
            y = (C_t[:, t] * s).sum(dim=-1)         # (B, D)
            outputs.append(y)

        y = torch.stack(outputs, dim=1)             # (B, L, D)

        


        #norm distribution
        y_row_norms= torch.sqrt(torch.mean(y**2, dim=2, keepdim=True) + 1e-6)
        y_row_norms=torch.sigmoid(y_row_norms)
        #multiply each unit y_t by his relevance
        y=y*y_row_norms
        #since norms serve as a sort of gate, we remove gate proj
        

        # ---- 4. Gating + output ----
        #y = y * gate
        y = self.out_proj(y)

        return y,y_row_norms


In [6]:

class MambaVideoConfig(PretrainedConfig):
    model_type = "mamba_video"

    def __init__(
        self,
        d_model=576,
        d_state=32,
        vit_name="timm/tiny_vit_21m_224.dist_in22k_ft_in1k",
        **kwargs
    ):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.d_state = d_state
        self.vit_name = vit_name
        

In [21]:


#config d_state,d_model,vitmodel

class Mambavideo(PreTrainedModel):
    def __init__(self,config):
        super().__init__(config)
        lorac=LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["qkv"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["head"],
)
        
        self.vittinylora=get_peft_model(AutoModel.from_pretrained(config.vit_name),lorac)
        self.mamba1=BidirectionalMambaLayer(config.d_model,config.d_state)
        self.mamba2=BidirectionalMambaLayer(config.d_model,config.d_state)
        self.mamba3=BidirectionalMambaLayer(config.d_model,config.d_state)
        self.post_init()
    def forward(self,x):
        x=self.vittinylora(**x).last_hidden_state
        #print(x.shape)
        x=x.mean(dim=(2, 3)).unsqueeze(0)
        
        x,weight1=checkpoint(self.mamba1,x, use_reentrant=False)
        #print(x.shape)
        x,weight2=checkpoint(self.mamba2,x, use_reentrant=False)
        x,weight3=checkpoint(self.mamba3,x, use_reentrant=False)
    
        return x,[weight1.view(-1),weight2.view(-1),weight3.view(-1)]

    
        

In [ ]:
#for name, module in model.named_modules():
    #print(name, type(module))

In [22]:
configur=MambaVideoConfig()
der_model=Mambavideo(configur).to("cuda")

In [ ]:
der_model.training

In [ ]:
der_model.save_pretrained("loo")

In [ ]:
der_model.eval()

In [ ]:
trainable_params = sum(
    p.numel() for p in der_model.parameters() if p.requires_grad
)


total_params = sum(p.numel() for p in der_model.parameters())


In [ ]:
der_model.vittinylora.training

In [ ]:
print(trainable_params)

In [ ]:
total_params

In [ ]:
der_model.eval()

In [10]:
def prototype_contrastive_loss(zi, pos_set, neg_set, tau=0.07):
    zi = nn.functional.normalize(zi, dim=-1)

    pos =nn.functional.normalize(pos_set.mean(dim=0), dim=-1)
    neg = nn.functional.normalize(neg_set.mean(dim=0), dim=-1)

    sp = (zi @ pos) / tau
    sn = (zi @ neg) / tau

    return nn.functional.softplus(sn - sp)

In [11]:
def getframes(dire,rangef):
    cap=cv2.VideoCapture(f"/kaggle/input/ucf101-action-recognition/{dire}")
    frames=cap.get(cv2.CAP_PROP_FRAME_COUNT)
    indexes=np.linspace(0,frames,random.randint(rangef[0],rangef[1]),dtype=np.int32)
    framesl=[]
    for d in indexes:
        cap.set(cv2.CAP_PROP_POS_FRAMES, d)
        ret, frame = cap.read()
    
        if not ret:
         continue
        framesl.append(Image.fromarray(frame))
    cap.release()
    return framesl
    
        
    
    

In [12]:
cossim=nn.CosineSimilarity(dim=0)

In [14]:
from itertools import chain

In [23]:
#tr loop

optimizer=torch.optim.Adam(lr=1e-4,params=der_model.parameters(),eps=1e-5)
#loss=likelihoodloss
grad_acc=16
evalst=16
classes=list(pd.unique(train["label"]))
it=20
cossim=nn.CosineSimilarity(dim=0)
evalloss=[]
alpv=1
optimizer.zero_grad()
for i in range(it):
    positive,negative=tuple(random.sample(classes,2))
    print(positive)
    positive=train[train["label"]==positive].sample(n=4)
    negative=train[train["label"]==negative].sample(n=3)
    
    
    vidrespos=[getframes(t,[8,16]) for t in positive["clip_path"].values]
    
    vidresneg=[getframes(t,[8,16]) for t in negative["clip_path"].values]
    
    vidrespos=[der_model(image_processor(t).to("cuda")) for t in vidrespos if len(t)>0]
    vidresneg=[der_model(image_processor(t).to("cuda")) for t in vidresneg if len(t)>0]
    vectorsp=[t[0][0,-1:,:] for t in vidrespos]
    weightsp=[t[1] for t in vidrespos]
    vectorsn=[t[0][0,-1:,:] for t in vidresneg]
    weightsn=[t[1] for t in vidresneg]
    weightst=list(chain.from_iterable(weightsp+weightsn))
    varianceloss=-alpv*sum([torch.var(t,dim=0).mean() for t in weightst])/len(weightst)
    print(varianceloss)
    
    
    
    if len(vidrespos)==0 or len(vidresneg)==0:
        print("empty")
        continue
    vidrespos=torch.stack(vectorsp).squeeze(1)
    vidresneg=torch.stack(vectorsn).squeeze(1)
    print(vidrespos.shape)
    zi,vidrespos=vidrespos[0,:].unsqueeze(0),vidrespos[1:,:]
    
    
    lossv=(prototype_contrastive_loss(zi,vidrespos,vidresneg)+ varianceloss)/grad_acc
    lossv.backward()
    if (i+1) % grad_acc==0:
        torch.nn.utils.clip_grad_norm_(der_model.parameters(),max_norm=1)
        optimizer.step()
        #for name, param in der_model.named_parameters():
         #if param.grad is not None:
        # Print the mean or norm to avoid flooding your screen with numbers
          #print(param.grad)
          #print(f"{name}: No gradient (frozen or disconnected)")
        optimizer.zero_grad()
        #eval
        evlm=[]
        #der_model.eval()
        for w in range(evalst):
         positive,negative=tuple(random.sample(classes,2))
         print(positive)
         positive=evalr[evalr["label"]==positive].sample(n=4)
         negative=evalr[evalr["label"]==negative].sample(n=3)
    
    
         vidrespos=[getframes(t,[8,16]) for t in positive["clip_path"].values]
         
         vidresneg=[getframes(t,[8,16]) for t in negative["clip_path"].values]
         
         print(len(weightst))
         with torch.no_grad():
          vidrespos=[der_model(image_processor(t).to("cuda")) for t in vidrespos if len(t)>0]
          vidresneg=[der_model(image_processor(t).to("cuda")) for t in vidresneg if len(t)>0]
         vectorsp=[t[0][0,-1:,:] for t in vidrespos]
         weightsp=[t[1] for t in vidrespos]
         vectorsn=[t[0][0,-1:,:] for t in vidresneg]
         weightsn=[t[1] for t in vidresneg]
         weightst=list(chain.from_iterable(weightsp+weightsn))
         varianceloss=-alpv*sum([torch.var(t,dim=0).mean() for t in weightst])/len(weightst)
    
         if len(vidrespos)==0 or len(vidresneg)==0:
             print("empty eval")
             continue
         vidrespos=torch.stack(vectorsp).squeeze(1)
         vidresneg=torch.stack(vectorsn).squeeze(1)
         zi,vidrespos=vidrespos[0,:].unsqueeze(0),vidrespos[1:,:]
        
         
         lossv=prototype_contrastive_loss(zi,vidrespos,vidresneg)/evalst
         evlm.append(lossv.cpu().numpy())
        evlm=sum(evlm)
        evalloss.append(evlm)
        plt.plot(evalloss)
        plt.show()
        #der_model.train()
        
    
               
        
        
    
    
    





Billiards
torch.Size([15, 576, 7, 7])
torch.Size([1, 15, 576])
torch.Size([9, 576, 7, 7])
torch.Size([1, 9, 576])
torch.Size([11, 576, 7, 7])
torch.Size([1, 11, 576])
torch.Size([15, 576, 7, 7])
torch.Size([1, 15, 576])
torch.Size([11, 576, 7, 7])
torch.Size([1, 11, 576])
torch.Size([14, 576, 7, 7])
torch.Size([1, 14, 576])
torch.Size([9, 576, 7, 7])
torch.Size([1, 9, 576])
tensor(-8.9289e-08, device='cuda:0', grad_fn=<DivBackward0>)
torch.Size([4, 576])
Kayaking
torch.Size([12, 576, 7, 7])
torch.Size([1, 12, 576])
torch.Size([9, 576, 7, 7])
torch.Size([1, 9, 576])
torch.Size([13, 576, 7, 7])
torch.Size([1, 13, 576])
torch.Size([8, 576, 7, 7])
torch.Size([1, 8, 576])
torch.Size([7, 576, 7, 7])
torch.Size([1, 7, 576])
torch.Size([14, 576, 7, 7])
torch.Size([1, 14, 576])
torch.Size([14, 576, 7, 7])
torch.Size([1, 14, 576])
tensor(-1.4974e-09, device='cuda:0', grad_fn=<DivBackward0>)
torch.Size([4, 576])
Skijet
torch.Size([9, 576, 7, 7])
torch.Size([1, 9, 576])
torch.Size([11, 576, 7, 7])

KeyboardInterrupt: 

In [ ]:
vidrespos[0].shape


In [ ]:
def getframes(dire,rangef):
    cap=cv2.VideoCapture(f"{dire}")
    frames=cap.get(cv2.CAP_PROP_FRAME_COUNT)
    indexes=np.linspace(0,frames,random.randint(rangef[0],rangef[1]),dtype=np.int32)
    framesl=[]
    for d in indexes:
        cap.set(cv2.CAP_PROP_POS_FRAMES, d)
        ret, frame = cap.read()
    
        if not ret:
         continue
        framesl.append(Image.fromarray(frame))
    cap.release()
    return framesl

In [ ]:
zoo=getframes("/kaggle/input/craimd/YTDown.com_Shorts_Footage-of-jewellery-store-robbery_Media_GjaU1WDBO-c_001_1080p.mp4",[16,30])
zooan=getframes("/kaggle/input/vidios/YTDown.com_YouTube_Me-at-the-zoo_Media_jNQXAC9IVRw_001_240p.mp4",[16,30])


In [ ]:
der_model.eval()

In [ ]:
zoo=image_processor(zoo).to("cuda")
zooan=image_processor(zooan).to("cuda")
with torch.no_grad():
    zoov=nn.functional.normalize(der_model2(zoo)[0,-1,:],dim=-1)
    zooanv=nn.functional.normalize(der_model2(zooan)[0,-1,:],dim=-1)
    

In [ ]:
zooan[6]

In [ ]:
der_model.training

In [ ]:
config = MambaVideoConfig.from_pretrained("/kaggle/working/lol")
der_model2 = Mambavideo.from_pretrained("/kaggle/working/lol", config=config)

In [ ]:
der_model2.training

In [ ]:
cossim(zoov,zooanv)

In [ ]:
next(der_model.mamba1.parameters()).device

In [ ]:
len(train[train["label"]=="Hammering"])

In [ ]:
vidrespos.shape

In [ ]:
vidresneg[2]